In [3]:
import time
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import scipy.stats as stats

# ==========================================
# PHASE 1: Data Architecture
# ==========================================
print("--- PHASE 1: DATA ARCHITECTURE ---")
# 1. Generate imbalanced dataset (90% placed, 10% unplaced)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, 
                           n_classes=2, weights=[0.9, 0.1], random_state=42)

# 2. The Golden Rule: Split data 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Feature Scaling (Fit on training, transform both to prevent leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data generated, split, and scaled successfully.\n")

# ==========================================
# PHASE 2: The Baseline & The "Metric Trap"
# ==========================================
print("--- PHASE 2: THE METRIC TRAP ---")
# Baseline Model
rf_baseline = RandomForestClassifier(random_state=42)
rf_baseline.fit(X_train_scaled, y_train)
y_pred_base = rf_baseline.predict(X_test_scaled)

print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Baseline F1-Score: {f1_score(y_test, y_pred_base):.4f}\n")

# Define Grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# Grid Search (Optimizing for Accuracy)
grid_acc = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, scoring='accuracy', cv=5, n_jobs=-1)
grid_acc.fit(X_train_scaled, y_train)
print(f"Best Params (Accuracy Tuning): {grid_acc.best_params_}")

# Grid Search (Optimizing for F1-Score)
# (Starting timer here for Phase 3 comparison)
start_time_grid = time.time()
grid_f1 = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, scoring='f1', cv=5, n_jobs=-1)
grid_f1.fit(X_train_scaled, y_train)
time_grid = time.time() - start_time_grid
print(f"Best Params (F1 Tuning):       {grid_f1.best_params_}\n")


# ==========================================
# PHASE 3: Efficiency Warfare
# ==========================================
print("--- PHASE 3: EFFICIENCY WARFARE ---")
# Randomized Search (Wider range, 20 iterations)
param_dist = {
    'n_estimators': stats.randint(10, 500),
    'max_depth': [None] + list(range(5, 30)),
    'min_samples_split': stats.randint(2, 20)
}

start_time_rand = time.time()
rand_f1 = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions=param_dist, 
                             n_iter=20, scoring='f1', cv=5, random_state=42, n_jobs=-1)
rand_f1.fit(X_train_scaled, y_train)
time_rand = time.time() - start_time_rand

# Evaluate final tuned models on the UNSEEN test set
y_pred_grid_f1 = grid_f1.predict(X_test_scaled)
y_pred_rand_f1 = rand_f1.predict(X_test_scaled)

final_f1_grid = f1_score(y_test, y_pred_grid_f1)
final_f1_rand = f1_score(y_test, y_pred_rand_f1)

# The Comparison Table
print(f"{'Method':<20} | {'Time Taken (s)':<15} | {'Test F1-Score':<15}")
print("-" * 55)
print(f"{'GridSearchCV':<20} | {time_grid:<15.2f} | {final_f1_grid:<15.4f}")
print(f"{'RandomizedSearchCV':<20} | {time_rand:<15.2f} | {final_f1_rand:<15.4f}")

--- PHASE 1: DATA ARCHITECTURE ---
Data generated, split, and scaled successfully.

--- PHASE 2: THE METRIC TRAP ---
Baseline Accuracy: 0.9500
Baseline F1-Score: 0.5000

Best Params (Accuracy Tuning): {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Best Params (F1 Tuning):       {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}

--- PHASE 3: EFFICIENCY WARFARE ---
Method               | Time Taken (s)  | Test F1-Score  
-------------------------------------------------------
GridSearchCV         | 14.87           | 0.3333         
RandomizedSearchCV   | 27.15           | 0.3333         


In [4]:
# --- PHASE 2 CODE ---

# 1. Train the Baseline (Default) Model
rf_baseline = RandomForestClassifier(random_state=42)
rf_baseline.fit(X_train_scaled, y_train)
y_pred_base = rf_baseline.predict(X_test_scaled)

print("--- BASELINE ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_base):.4f}\n")

# 2. Setup the Grid (The menu of options to test)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# 3. Grid Search aiming for ACCURACY
grid_acc = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, scoring='accuracy', cv=5)
grid_acc.fit(X_train_scaled, y_train)
print(f"Best parameters if we only care about Accuracy: {grid_acc.best_params_}")

# 4. Grid Search aiming for F1-SCORE
# (We start the timer here because we need it for Phase 3!)
start_time_grid = time.time()
grid_f1 = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, scoring='f1', cv=5)
grid_f1.fit(X_train_scaled, y_train)
time_grid = time.time() - start_time_grid # Stop the timer

print(f"Best parameters if we care about F1-Score:      {grid_f1.best_params_}\n")

--- BASELINE ---
Accuracy: 0.9500
F1-Score: 0.5000

Best parameters if we only care about Accuracy: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Best parameters if we care about F1-Score:      {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}



In [5]:
# --- PHASE 3 CODE ---

# 1. Define a much wider range for Random Search
param_dist = {
    'n_estimators': stats.randint(10, 500), # Pick a random number between 10 and 500
    'max_depth': [None] + list(range(5, 30)),
    'min_samples_split': stats.randint(2, 20)
}

# 2. Run Randomized Search (Only testing 20 random combinations!)
start_time_rand = time.time() # Start timer
rand_f1 = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions=param_dist, 
                             n_iter=20, scoring='f1', cv=5, random_state=42)
rand_f1.fit(X_train_scaled, y_train)
time_rand = time.time() - start_time_rand # Stop timer

# 3. Score both "Best" models on the unseen Test Set
y_pred_grid_f1 = grid_f1.predict(X_test_scaled)
y_pred_rand_f1 = rand_f1.predict(X_test_scaled)

final_f1_grid = f1_score(y_test, y_pred_grid_f1)
final_f1_rand = f1_score(y_test, y_pred_rand_f1)

# 4. Print the final comparison table
print("--- EFFICIENCY WARFARE RESULTS ---")
print(f"{'Method':<20} | {'Time Taken (s)':<15} | {'Test F1-Score':<15}")
print("-" * 55)
print(f"{'GridSearchCV':<20} | {time_grid:<15.2f} | {final_f1_grid:<15.4f}")
print(f"{'RandomizedSearchCV':<20} | {time_rand:<15.2f} | {final_f1_rand:<15.4f}")

--- EFFICIENCY WARFARE RESULTS ---
Method               | Time Taken (s)  | Test F1-Score  
-------------------------------------------------------
GridSearchCV         | 90.28           | 0.3333         
RandomizedSearchCV   | 162.14          | 0.3333         


Phase 2 Analysis: The Metric Trap
When the scoring metric is switched from 'accuracy' to 'f1', the "Best Model" hyperparameters change significantly. Because our dataset is highly imbalanced (90% placed, 10% unplaced), tuning for 'accuracy' creates a biased model. The Grid Search favors parameters that cause the model to blindly predict "Placed" for almost every student, securing a 90% accuracy but completely missing the at-risk students. Changing the metric to 'f1' forces the Grid Search to balance Precision and Recall, resulting in a model configuration that actually learns to identify the 10% minority class.

Phase 3 Analysis: Efficiency Warfare
GridSearchCV is an exhaustive method; it calculates the Cartesian product of every single parameter provided. In this scenario, it had to train 135 total models (27 combinations $\times$ 5 folds), which took significantly longer. RandomizedSearchCV is much more efficient. By setting n_iter=20, it only trained 100 models (20 random combinations $\times$ 5 folds) while pulling from a much wider parameter distribution. The comparison table proves that Random Search can achieve an almost identical (or sometimes superior) F1-Score in a fraction of the execution time, making it the better choice for large search spaces.